In [71]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor, Pool

In [72]:
# import csv files

train = pd.read_csv('/Users/glenharris/accident_risk/data/train.csv')
test = pd.read_csv('/Users/glenharris/accident_risk/data/test.csv')

In [73]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517754 entries, 0 to 517753
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      517754 non-null  int64  
 1   road_type               517754 non-null  object 
 2   num_lanes               517754 non-null  int64  
 3   curvature               517754 non-null  float64
 4   speed_limit             517754 non-null  int64  
 5   lighting                517754 non-null  object 
 6   weather                 517754 non-null  object 
 7   road_signs_present      517754 non-null  bool   
 8   public_road             517754 non-null  bool   
 9   time_of_day             517754 non-null  object 
 10  holiday                 517754 non-null  bool   
 11  school_season           517754 non-null  bool   
 12  num_reported_accidents  517754 non-null  int64  
 13  accident_risk           517754 non-null  float64
dtypes: bool(4), float64(

In [74]:
#Drop ID column since it is an identifier without meaningful info, therefore not a feature (leave it in the training df since it is needed to match predictions)

train = train.drop(columns = ['id'])


In [75]:
# split the dataframe into features and target DF

X = train.drop(['accident_risk'], axis=1)
y = train['accident_risk'].copy()

In [76]:
#Identify Categorical columns 

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print("Categorical features:", categorical_cols)

Categorical features: ['road_type', 'lighting', 'weather', 'time_of_day']


In [77]:
# split into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

In [78]:
#Create Pools for CatBoost 

train_pool = Pool(X_train, y_train, cat_features=categorical_cols)
test_pool = Pool(X_test, y_test, cat_features=categorical_cols)

In [79]:
#Hyperparameter Tuning (CatBoost has built in grid/random search)

#Create Grid

param_dist = {
    "learning_rate": (0.01, 0.15),
    "iterations": [1500, 2000, 2500],
    "depth": [4, 6, 8, 10],
    "l2_leaf_reg": (1, 30),
    "bagging_temperature": (0, 5),
    "random_strength": (1, 20),
    "subsample": (0.7, 1.0)
}

In [80]:
#Create model, tune before fit, do not set parameters on how the model learns since tuning will do this

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=200
)

In [82]:
#Complete hyperparameter grid or random search. If you use random search you need n_iter= a number. (not needed for grid)

grid_search_results = model.grid_search(
    param_dist,
    X=train_pool,
    y=None, #Pool already contains y
    cv=3,
    verbose=1
)

0:	learn: 0.3863639	test: 0.3862971	best: 0.3862971 (0)	total: 88.9ms	remaining: 2m 13s
200:	learn: 0.0845529	test: 0.0846711	best: 0.0846711 (200)	total: 5.83s	remaining: 37.7s
400:	learn: 0.0582097	test: 0.0583724	best: 0.0583724 (400)	total: 11.4s	remaining: 31.2s
600:	learn: 0.0569523	test: 0.0570718	best: 0.0570718 (600)	total: 17.8s	remaining: 26.6s
800:	learn: 0.0567635	test: 0.0568705	best: 0.0568705 (800)	total: 24s	remaining: 20.9s
1000:	learn: 0.0567009	test: 0.0568070	best: 0.0568070 (1000)	total: 29.7s	remaining: 14.8s
1200:	learn: 0.0566583	test: 0.0567651	best: 0.0567651 (1200)	total: 35.6s	remaining: 8.87s
1400:	learn: 0.0566183	test: 0.0567250	best: 0.0567250 (1400)	total: 41.7s	remaining: 2.94s
1499:	learn: 0.0565972	test: 0.0567039	best: 0.0567039 (1499)	total: 44.7s	remaining: 0us

bestTest = 0.0567039329
bestIteration = 1499

0:	loss: 0.0567039	best: 0.0567039 (0)	total: 45s	remaining: 4h 47m 25s
0:	learn: 0.3366124	test: 0.3365334	best: 0.3365334 (0)	total: 45.7ms

In [87]:
#Extract the best parameters and review

best_params = grid_search_results["params"]
print("Best Params:", best_params)

Best Params: {'bagging_temperature': 0, 'subsample': 0.7, 'random_strength': 1, 'depth': 6, 'learning_rate': 0.15, 'l2_leaf_reg': 1, 'iterations': 1500}


In [88]:
#Create a new model with the best parameters from tuning

final_model = CatBoostRegressor(
    **best_params,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

#Fit model

final_model.fit(train_pool, eval_set=test_pool)

0:	learn: 0.1468164	test: 0.1465604	best: 0.1465604 (0)	total: 138ms	remaining: 3m 27s
100:	learn: 0.0563104	test: 0.0565813	best: 0.0565813 (100)	total: 5.99s	remaining: 1m 22s
200:	learn: 0.0560756	test: 0.0564166	best: 0.0564166 (200)	total: 11.8s	remaining: 1m 16s
300:	learn: 0.0559624	test: 0.0563514	best: 0.0563513 (298)	total: 17.6s	remaining: 1m 9s
400:	learn: 0.0558524	test: 0.0562887	best: 0.0562887 (400)	total: 23.2s	remaining: 1m 3s
500:	learn: 0.0557836	test: 0.0562629	best: 0.0562629 (500)	total: 28.7s	remaining: 57.2s
600:	learn: 0.0557117	test: 0.0562437	best: 0.0562437 (600)	total: 34.6s	remaining: 51.7s
700:	learn: 0.0556553	test: 0.0562284	best: 0.0562282 (694)	total: 40.4s	remaining: 46.1s
800:	learn: 0.0555965	test: 0.0562237	best: 0.0562237 (800)	total: 46.8s	remaining: 40.9s
900:	learn: 0.0555420	test: 0.0562160	best: 0.0562148 (878)	total: 52.7s	remaining: 35s
1000:	learn: 0.0554969	test: 0.0562130	best: 0.0562113 (989)	total: 58.7s	remaining: 29.3s
1100:	learn:

In [89]:
#Evaluate the model

#Use test data to create predictions

val_pred = final_model.predict(X_test)

# CLIP to 0–1 since regression can output slightly outside the range and we are predicting probability 
val_pred = np.clip(val_pred, 0, 1)

#Find RMSE score using the y test data compared to the prediction data

rmse = root_mean_squared_error(y_test, val_pred)
print("Validation RMSE:", rmse)

Validation RMSE: 0.05620193972917087


In [90]:
#Predict on the Kaggle test set (previously uploaded csv to test)

#save the test IDs (if they exist), so that once we remove them for testing we can add them back to the prediction results for Kaggle submission
test_ids = test["id"] if "id" in test.columns else None

#Create a df without ID since the model was created without ID col. 
test_features = test.drop(columns=["id"], errors="ignore")

#Generate prediction and clip the results to be between 0 and 1. 
test_pred = model.predict(test_features)
test_pred = np.clip(test_pred, 0, 1)

In [91]:
#Build submission dataframe (this is where IDs are reattached)

submission = pd.DataFrame({
    "id": test_ids,
    "accident_risk": test_pred
})

In [92]:
#Save CSV 

submission.to_csv("submission.csv", index=False)